# Clean `emr_cycle_aspirations`

A first-pass cleaning of the `emr_cycle_aspirations` table (3228 rows, 40 columns) before analysis.

What this notebook does, in order:
1. **Profile** every column (how full it is, how many distinct values, its type)
2. **Drop** columns that are completely empty
3. **Keep** selected subset of columns
4. **Parse** the date columns (currently stored as text)
5. **Tidy** column types
6. **Save** a cleaned copy + keep an audit note of what changed

The same `profile()` function works on the other `emr_*` tables too, so you can reuse this pattern.

> **Before running:** finish the venv setup in this repo and install the packages — open a terminal and run `pip install pandas pyarrow`. The `requirements.txt` alongside this notebook lists everything.
>
> **Important:** keep your raw patient data in a `data/` folder that is git-ignored. Don't commit EMR exports to GitHub, even a private repo.

## Setup

In [1]:
from pathlib import Path
import pandas as pd

In [12]:
# --- point this at your raw export ---
PATH = Path("../data/emr_cycle_aspirations.csv")

# loading data
df = pd.read_csv(PATH)


In [6]:
#setting display options to show all columns and full width
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)        # Allow full width

In [7]:
print(df.shape)

(3228, 40)


## 1. Profile every column

One row per column so you can see at a glance what's worth keeping.

In [8]:
# Already know there are many all NULL columns, so dropping those first
print(df.shape)
df = df.dropna(axis=1, how="all")
print(df.shape)

(3228, 40)
(3228, 29)


In [10]:
def profile(frame: pd.DataFrame) -> pd.DataFrame:
    """One row per column: how full it is, how many distinct values, its dtype,
    and whether all non-null values are identical."""
    out = pd.DataFrame({
        "non_null": frame.notna().sum(),
        "nulls": frame.isna().sum(),
        "distinct": frame.nunique(dropna=True),
        "dtype": frame.dtypes.astype(str),
    })

    out["pct_null"] = (out["nulls"] / len(frame) * 100).round(1)

    # True if all non-null values in the column are the same
    out["all_values_same"] = out["distinct"] <= 1

    return out.sort_values("non_null", ascending=False)

prof = profile(df)
prof

# patid = Registry System ID in nAble Patient PM
# sart_id = SART Cycle ID in nAble (this is the only column that is directly searchable in nAble)
# planned_eggsource = Account number in nAble Patient PM
# partner_id -> Registry System ID in nAble Patient PM for the partner
# planned_spermsource -> Account number in nAble Patient PM for the partner

,non_null,nulls,distinct,dtype,pct_null,all_values_same
id,3228,0,3228,str,0.0,False
addedby,3228,0,14,int64,0.0,False
addedon,3228,0,3228,str,0.0,False
for_position,3228,0,1,str,0.0,True
cycleid,3228,0,3220,str,0.0,False
technique,3228,0,1,str,0.0,True
retrieved,3228,0,3220,str,0.0,False
patid,3228,0,2404,int64,0.0,False
denuding_type,3228,0,1,str,0.0,True
oocytes_retrieved,3227,1,52,float64,0.0,False


## 2. Drop completely empty columns and columns with only one value

These have zero informative values, so there's nothing to lose.

In [13]:
# dropping columns with all the same value
cols_to_drop = prof.index[
    (prof["non_null"] == 0) | (prof["all_values_same"])
].tolist()

print(f"Dropping {len(cols_to_drop)} empty or constant columns:\n")
for c in cols_to_drop:
    print("  -", c)

df = df.drop(columns=cols_to_drop)

print(f"\nRemaining: {df.shape[1]} columns")

Dropping 4 empty or constant columns:

  - for_position
  - technique
  - denuding_type
  - med_given

Remaining: 36 columns


## 3. Keeping subset of columns

Only looking to keep columns that inform response type. Filtering beyond the NA and single value columns

In [14]:
# print list of columns with their data types and percentage of nulls
print("\nColumn summary:")
for col in df.columns:
    non_null = df[col].notna().sum()
    total = len(df)
    pct_null = (total - non_null) / total * 100
    dtype = df[col].dtype
    print(f"{col}: {dtype}, {pct_null:.1f}% null")


Column summary:
id: str, 0.0% null
addedby: int64, 0.0% null
addedon: str, 0.0% null
deletedby: float64, 99.7% null
deletedon: str, 99.7% null
cycleid: str, 0.0% null
patid: int64, 0.0% null
donorid: float64, 100.0% null
retrieved: str, 0.0% null
retrievedby: float64, 0.3% null
oocytes_retrieved: float64, 0.0% null
complication: str, 99.9% null
note: str, 72.6% null
follicles_collected: float64, 100.0% null
follicles_right_collected: float64, 100.0% null
follicles_left_collected: float64, 100.0% null
follicles_washed: float64, 100.0% null
follicle_lead_size: float64, 100.0% null
denuding_performed: str, 1.2% null
denuding_performedby: float64, 1.6% null
patient_identifiedby: float64, 3.8% null
retrievedtech: float64, 0.4% null
spermwitness: float64, 99.8% null
hospitalization_required: float64, 100.0% null
importid: float64, 100.0% null
labassistant: float64, 0.7% null
scrub_tech: float64, 3.2% null
crna: str, 1.1% null
needle_type: str, 1.0% null
duration: str, 1.6% null
accession_nu

In [15]:
# create a list of columns that I want to keep
keep_cols = [
    "cycleid", "patid", "donorid", "retrieved", "retrievedby", "oocytes_retrieved", 
    "complication", "note", "denuding_performed", "denuding_performedby", "retrievedtech", 
    "labassistant", "scrub_tech", "crna", "needle_type", "duration", "accession_number" 
]
# filter the dataframe to keep only those columns
df = df[keep_cols]
print(f"\nAfter filtering to {len(keep_cols)} columns: {df.shape[1]} columns")


After filtering to 17 columns: 17 columns


In [18]:
print(df.shape)

(3228, 17)


In [19]:
# save as csv
df.to_csv("../data/emr_cycle_aspirations_filtered.csv", index=False)